# NIRN.Ai — GR Corpus Indexing (Kaggle, 2x T4)

Builds `index.faiss`, `chunks.pkl`, and `metadata.json` for the full Maharashtra GR corpus
(`orgpedia/mahGRs` — 33 departments, ~98,929 GRs, 197,858 Marathi+English text files, ~2.95M
clause-level chunks).

Output is a **drop-in replacement** for `backend/data/` — copy the three output files over the
existing ones and the backend picks them up with no code changes to the load path.

**Kaggle notebook settings required:**
- Accelerator: **GPU T4 x2**
- Internet: **On** (needed to clone the GitHub repo and download the embedding model)

**Where the output lands:**
- Working files during the run: `/kaggle/working/backend_data/index.faiss`, `chunks.pkl`, `metadata.json` (deleted once zipped, see Section 12)
- Final downloadable archive: `/kaggle/working/backend_data.zip` — visible in the notebook's **Output** tab after the run finishes. Download it, unzip, and drop the three files into `backend/data/` locally.

**Estimated end-to-end time:** a first full run took ~2 hours with the original sequential
parse-then-embed design. The current version pipelines parsing and embedding (Section 7–8) and
uses a larger batch size, targeting roughly **half that — around 1 hour** — on a 2x T4 session.
This is a target, not a guarantee: exact time depends on Kaggle's momentary T4 allocation and is
dominated by the embedding step regardless.

Comfortably within Kaggle's 9-hour session limit and its free weekly GPU quota.

## Indexing strategy
1. **Full corpus, not a sample** — every GR, both languages (the current backend index only covers ~5% of GRs; this rebuild covers 100%).
2. **Clause-level chunking** instead of blind 500-char windows — splits on numbered clause boundaries so each chunk is a coherent unit of meaning, not a mid-sentence cut.
3. **Self-contained chunks** — every chunk gets the GR's header context (department, title, GR number, date) prepended, so a chunk retrieved on its own still carries enough context for the LLM to judge it correctly. This is what keeps prompts short (fast generation) while staying accurate (trustworthy conflict checks).
4. **Real metadata extracted from the OCR header block** (regex, deterministic, bilingual) — `title`, `gr_number`, `issued_on`, `cited_references`. The current index has none of these; `retrieval.py` already reads `title`/`issued_on` via `.get()` with fallback, so this just fills in real values instead of placeholders.
5. **`cited_references`** — the GR numbers each resolution's own "Read—" / "वाचा" section cites, extracted with the *same* regex/format used for the file's own `gr_number`. This gives conflict detection an explicit citation trail (does draft clause X relate to a GR that a later GR already amended/superseded?) instead of relying on embedding similarity alone to guess relationships.
6. **Why this hits the speed targets locally (M4, Gemma3:4b):** FAISS search itself is sub-second at this corpus size regardless of chunk count — the real lever for the 1-minute draft / 30-second conflict budgets is *prompt size*. Clause-sized, self-contained chunks keep each retrieved hit small and information-dense, so `TOP_K`/`CANDIDATES_PER_CLAUSE` retrieved chunks stay well within a fast local-generation prompt budget.

## Performance notes (Kaggle 2x T4)
- **CPU parsing overlapped with GPU embedding** (Section 7–8) — a background thread parses department *N+1* while the GPUs embed department *N*, instead of running two fully sequential phases. This removes what was previously dead GPU-idle time from the critical path.
- **Persistent CPU worker pool** for parsing, reused across all 33 departments, with `chunksize` scaled per department size — avoids both repeated process-spawn overhead and IPC overhead on large departments.
- **`model.max_seq_length = 256`** — the default 512-token cap wastes GPU compute padding every batch for chunks that are only ever a few hundred characters.
- **`BATCH_SIZE = 512`** with fp16 weights — bumped after the actual run showed only ~3GB/15GB used per T4 at batch 256, meaning there was headroom for a bigger batch.
- **Per-department checkpointing for both parsing and embedding** — a crash now only costs the department in progress, not the whole pass (an earlier run OOM'd on CPU RAM after ~2 hours and lost all embedding progress because only parsing was checkpointed at the time).
- **`faiss.omp_set_num_threads`** (Section 9) — uses all available CPU cores for index construction.
- **Streaming zip with immediate delete** (Section 12) — avoids holding two full copies of the multi-GB output on disk at once, relevant given Kaggle's 20GB `/kaggle/working` output cap.

## 1. Install dependencies

In [ ]:
!pip install -q faiss-cpu sentence-transformers python-dateutil tqdm

## 2. Clone the GR corpus from GitHub

In [ ]:
%cd /kaggle/working
!rm -rf mahGRs
!git clone --depth 1 https://github.com/orgpedia/mahGRs.git
# Some orgpedia mah* repos store large text trees via Git LFS — pull explicitly in case.
!cd mahGRs && (git lfs pull || true)
!find mahGRs/GRs -name '*.txt' | wc -l

## 3. Config

`MAX_CHARS`/`OVERLAP` match the backend's existing `CHUNK_CHARS`/`CHUNK_OVERLAP` defaults
(`backend/config.py`) so behaviour stays consistent with what the rest of the system assumes.

In [ ]:
import os

# Fork-safe: avoids HF tokenizers deadlock warnings once a model/tokenizer has been loaded
# in the parent process before ProcessPoolExecutor workers are spawned.
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_DIR = "/kaggle/working/mahGRs/GRs"
OUT_DIR = "/kaggle/working/backend_data"
SHARD_DIR = "/kaggle/working/shards"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(SHARD_DIR, exist_ok=True)

MAX_CHARS = 500
OVERLAP = 100

MODEL_NAME = "intfloat/multilingual-e5-base"
# 512 (up from 128, then 256) — the actual run showed only ~3GB/15GB used per T4 at batch 256,
# meaning there was plenty of headroom. Bigger batches on Turing-generation tensor cores
# (T4 = sm75) tend to keep achieving real throughput gains even when nvidia-smi already reports
# ~100% GPU-util%, since that figure is a coarse duty-cycle measure, not a saturation measure —
# larger GEMMs use the tensor cores more efficiently per active cycle. Watch for an OOM on this
# GPU (distinct from the earlier CPU-RAM OOM) if any batch contains unusually long chunks; drop
# back to 256 if so.
BATCH_SIZE = 512

print("Ready.")

## 4. Discover files

Pairs are not required to be matched up front — each file (mr and en) becomes its own set of
chunks tagged with `language`, exactly like the current schema.

In [ ]:
from pathlib import Path

discovered_files = []  # (filepath, gr_id, department, language)
for dept_dir in sorted(Path(REPO_DIR).iterdir()):
    if not dept_dir.is_dir():
        continue
    department = dept_dir.name
    for f in dept_dir.glob("*.txt"):
        gr_id = f.name.split(".")[0]
        language = "mr" if ".mr." in f.name else "en"
        discovered_files.append((str(f), gr_id, department, language))

print("Total files discovered:", len(discovered_files))
print("Expected: ~197,858 (98,929 GRs x 2 languages) — verify this is close before continuing.")

## 5. Bilingual header parsing

Deterministic regex over the fixed OCR header block every GR shares:
subject line → `Government of Maharashtra` / `महाराष्ट्र शासन` boilerplate → GR number line →
date line → `Read-` / `वाचा` citation lines → numbered clause body.

Any field that doesn't match falls back to `None` / `[]` — the backend already tolerates missing
`title`/`issued_on` via `.get()`, so a partial match on odd/damaged OCR is not a hard failure.

In [ ]:
import re
from dateutil import parser as dateparser

PAGE_MARKER_RE = re.compile(r'^#\s*Page\s*\d+\s*$', re.MULTILINE)

EN_BOILERPLATE_RE = re.compile(r'Government of Maharashtra', re.IGNORECASE)
MR_BOILERPLATE_RE = re.compile(r'महाराष्ट्र\s*शासन')

EN_GR_NO_RE = re.compile(r'Government Resolution No[.:]?\s*[:\-]?\s*(.+)', re.IGNORECASE)
MR_GR_NO_RE = re.compile(r'शासन\s*निर्णय\s*क्रमांक[ः:]?\s*(.+)')

EN_DATE_RE = re.compile(r'Date\s*[:\-]\s*(.+)', re.IGNORECASE)
MR_DATE_RE = re.compile(r'दिनांक\s*[:ः]\s*(.+)')

EN_READ_RE = re.compile(r'^Read[\s\-]*[:\-]?\s*(.+)', re.IGNORECASE)
MR_READ_RE = re.compile(r'^वाचा\s*[:ः]?\s*(.+)')

CLAUSE_SPLIT_RE = re.compile(r'\n\s*(?=\d{1,2}\.\s)')

DEVANAGARI_DIGITS = "०१२३४५६७८९"
MARATHI_MONTHS = {
    "जानेवारी": "January", "फेब्रुवारी": "February", "मार्च": "March", "एप्रिल": "April",
    "मे": "May", "जून": "June", "जुलै": "July", "ऑगस्ट": "August",
    "सप्टेंबर": "September", "ऑक्टोबर": "October", "नोव्हेंबर": "November", "डिसेंबर": "December",
}


def normalize_date(raw, language):
    if not raw:
        return None
    text = raw
    if language == "mr":
        for mr_month, en_month in MARATHI_MONTHS.items():
            text = text.replace(mr_month, en_month)
        for i, d in enumerate(DEVANAGARI_DIGITS):
            text = text.replace(d, str(i))
    try:
        return dateparser.parse(text, fuzzy=True, dayfirst=True).date().isoformat()
    except Exception:
        return None


def parse_header(text, language):
    clean = PAGE_MARKER_RE.sub("", text).strip()
    lines = [l.strip() for l in clean.splitlines() if l.strip()]

    boilerplate_re = MR_BOILERPLATE_RE if language == "mr" else EN_BOILERPLATE_RE
    gr_no_re = MR_GR_NO_RE if language == "mr" else EN_GR_NO_RE
    date_re = MR_DATE_RE if language == "mr" else EN_DATE_RE
    read_re = MR_READ_RE if language == "mr" else EN_READ_RE

    title_lines = []
    body_start_idx = len(lines)
    for i, line in enumerate(lines):
        if boilerplate_re.search(line):
            body_start_idx = i
            break
        title_lines.append(line)

    gr_number = None
    issued_on_raw = None
    references = []
    for line in lines[body_start_idx:]:
        if gr_number is None:
            m = gr_no_re.search(line)
            if m:
                gr_number = m.group(1).strip()
                continue
        if issued_on_raw is None:
            m = date_re.search(line)
            if m:
                issued_on_raw = m.group(1).strip()
                continue
        m = read_re.match(line)
        if m:
            references.append(m.group(1).strip())

    return {
        "title": " ".join(title_lines).strip() or None,
        "gr_number": gr_number,
        "issued_on": normalize_date(issued_on_raw, language),
        "cited_references": references,
    }

## 6. Clause-level chunking

Splits on numbered-clause boundaries (falls back to the whole body as one clause if no numbering
is detected), then further sub-splits any clause longer than `MAX_CHARS` with the same overlap the
backend already uses. Every resulting chunk gets a compact header line prepended so it stands on
its own for retrieval.

In [ ]:
def split_into_clauses(body_text):
    parts = CLAUSE_SPLIT_RE.split(body_text)
    parts = [p.strip() for p in parts if p.strip()]
    return parts if len(parts) > 1 else [body_text.strip()]


def enforce_max_chars(clause):
    if len(clause) <= MAX_CHARS:
        return [clause]
    out = []
    start = 0
    while start < len(clause):
        out.append(clause[start:start + MAX_CHARS])
        start += MAX_CHARS - OVERLAP
    return out


def process_file(args):
    filepath, gr_id, department, language = args
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            raw_text = f.read()
    except Exception:
        return []

    parsed = parse_header(raw_text, language)

    header_ctx = "[{dept} | {title} | {gr_no} | {date}]".format(
        dept=department.replace("_", " "),
        title=parsed["title"] or gr_id,
        gr_no=parsed["gr_number"] or "",
        date=parsed["issued_on"] or "",
    )

    clean_body = PAGE_MARKER_RE.sub("", raw_text).strip()
    clauses = split_into_clauses(clean_body)

    chunk_texts = []
    for clause in clauses:
        chunk_texts.extend(enforce_max_chars(clause))

    chunks = []
    for idx, ctext in enumerate(chunk_texts):
        chunks.append({
            "gr_id": gr_id,
            "department": department,
            "language": language,
            "chunk_id": idx,
            "text": f"{header_ctx}\n\n{ctext}",
            "title": parsed["title"],
            "gr_number": parsed["gr_number"],
            "issued_on": parsed["issued_on"],
            "cited_references": parsed["cited_references"],
        })
    return chunks

## 7–8. Parse + embed pipeline (CPU parsing overlapped with GPU embedding)

The previous version ran two fully sequential phases — parse ALL departments on CPU, *then*
embed ALL departments on GPU — which spent real wall-clock time with the GPUs sitting idle while
the CPU parsed. This version pipelines the two: while the GPU embeds department *N*, a background
thread parses department *N+1* on the CPU pool, so parsing time is hidden behind GPU embedding
time instead of adding to the total.

Also switches to the current, non-deprecated multi-process API (`model.encode(..., pool=pool)`) —
`encode_multi_process` has been folded into `encode` in the installed `sentence-transformers`
version — and bumps `BATCH_SIZE` (Section 3) since the GPUs showed only ~3GB/15GB used at the
previous batch size, meaning there was headroom for bigger batches (better tensor-core utilization
per active cycle, even at already-high GPU-util%).

Each department's chunks and embeddings are still written to disk immediately (`SHARD_DIR` /
`EMB_SHARD_DIR`) and skipped on re-run if already present — same crash-safety as before, now
covering both parsing and embedding.

In [ ]:
import gc
import pickle
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

EMB_SHARD_DIR = "/kaggle/working/emb_shards"
os.makedirs(EMB_SHARD_DIR, exist_ok=True)

files_by_dept = defaultdict(list)
for entry in discovered_files:
    files_by_dept[entry[2]].append(entry)
departments = list(files_by_dept.keys())

# Leave one CPU core free for the GPU workers' own tokenization overhead — running the parsing
# pool at every last core while embedding is happening concurrently would create contention that
# eats into the overlap benefit instead of adding to it.
PARSE_WORKERS = max(1, os.cpu_count() - 1)
print("CPU workers for parsing:", PARSE_WORKERS)

n_gpus = torch.cuda.device_count()
print("CUDA devices visible:", n_gpus)

model_kwargs = {"torch_dtype": torch.float16} if n_gpus > 0 else {}
model = SentenceTransformer(MODEL_NAME, model_kwargs=model_kwargs)

# The model defaults to a 512-token max sequence length. Our chunks are clause-sized (<=500
# chars + a short header line), so every batch would otherwise be padded out to 512 tokens for
# nothing — wasted compute on every single forward pass. Capping this to 256 is generous headroom
# for these chunk sizes and meaningfully cuts GPU time with no truncation in practice.
model.max_seq_length = 256

pool = None
if n_gpus >= 2:
    pool = model.start_multi_process_pool(target_devices=["cuda:0", "cuda:1"])
elif n_gpus == 1:
    model = model.to("cuda:0")


def embed_texts(texts):
    passage_texts = ["passage: " + t for t in texts]
    if pool is not None:
        # Current (non-deprecated) multi-process API: pass the pool straight to encode().
        return model.encode(passage_texts, pool=pool, batch_size=BATCH_SIZE)
    return model.encode(
        passage_texts,
        batch_size=BATCH_SIZE if n_gpus else 32,
        show_progress_bar=False,
        convert_to_numpy=True,
    )


def parse_department(department, cpu_pool):
    """Runs on the background thread; parses one department's files via the shared CPU pool."""
    shard_path = os.path.join(SHARD_DIR, f"{department}.pkl")
    if os.path.exists(shard_path):
        with open(shard_path, "rb") as f:
            return pickle.load(f)

    files = files_by_dept[department]
    chunksize = max(1, len(files) // (PARSE_WORKERS * 4))
    dept_chunks = []
    for result in cpu_pool.map(process_file, files, chunksize=chunksize):
        dept_chunks.extend(result)

    with open(shard_path, "wb") as f:
        pickle.dump(dept_chunks, f)
    return dept_chunks


with ProcessPoolExecutor(max_workers=PARSE_WORKERS) as cpu_pool, \
     ThreadPoolExecutor(max_workers=1) as prefetcher:

    # Kick off parsing for the first department before the loop starts.
    next_future = prefetcher.submit(parse_department, departments[0], cpu_pool)

    for i, department in enumerate(tqdm(departments, desc="departments")):
        dept_chunks = next_future.result()

        # Immediately queue up parsing for the *next* department so it runs on the CPU pool
        # while this department's chunks are being embedded on the GPUs below.
        if i + 1 < len(departments):
            next_future = prefetcher.submit(parse_department, departments[i + 1], cpu_pool)

        emb_shard_path = os.path.join(EMB_SHARD_DIR, f"{department}.npy")
        if os.path.exists(emb_shard_path):
            print(f"[skip] {department}: embeddings already on disk")
            del dept_chunks
            continue

        dept_embeddings = np.asarray(embed_texts([c["text"] for c in dept_chunks]), dtype="float32")
        np.save(emb_shard_path, dept_embeddings)
        print(f"[done] {department}: {len(dept_chunks)} chunks embedded")

        del dept_chunks, dept_embeddings
        gc.collect()

if pool is not None:
    model.stop_multi_process_pool(pool)

print("\nParse + embed pipeline complete.")

## 9. Build the FAISS index (merge department shards, streamed)

Loads one department's chunks + embeddings shard at a time, adds its vectors to the index, extends
the final chunk list, then **deletes that shard's files from disk immediately** (not just frees
the in-memory copy) before moving to the next.

This isn't just a RAM optimization — a prior run hit `RuntimeError: ... No space left on device`
writing the final `index.faiss` (needs ~8.4GB) because the `.npy` embedding shards it was merged
from (also ~8.4GB combined, since `index.faiss` is just those same vectors restacked) were still
sitting on disk at that point — the old code only deleted them *after* the write, which is too
late. Deleting each shard right after it's merged means the source embeddings never coexist on
disk with the index being built from them. The `mahGRs` clone (~1GB) is also no longer needed once
every department is parsed, so it's removed here too.

In [ ]:
import shutil

import faiss

faiss.omp_set_num_threads(os.cpu_count())

# No longer needed once every department is parsed — frees ~1GB before the write-heavy steps below.
shutil.rmtree("/kaggle/working/mahGRs", ignore_errors=True)

all_chunks = []
index = None

for department in files_by_dept.keys():
    chunks_shard_path = os.path.join(SHARD_DIR, f"{department}.pkl")
    emb_shard_path = os.path.join(EMB_SHARD_DIR, f"{department}.npy")

    with open(chunks_shard_path, "rb") as f:
        dept_chunks = pickle.load(f)
    dept_embeddings = np.load(emb_shard_path)
    faiss.normalize_L2(dept_embeddings)

    if index is None:
        dimension = dept_embeddings.shape[1]
        index = faiss.IndexFlatIP(dimension)

    index.add(dept_embeddings)
    all_chunks.extend(dept_chunks)

    del dept_chunks, dept_embeddings

    # Delete from disk now, not just the in-memory copy — index.faiss will need ~8GB+ of its
    # own to write out, and these source .npy files take up the same amount of space again if
    # left sitting around until after that write (this is what caused the earlier OOD-disk crash).
    os.remove(chunks_shard_path)
    os.remove(emb_shard_path)

print("Total chunks:", len(all_chunks))
print("Vectors in index:", index.ntotal)

shutil.rmtree(SHARD_DIR, ignore_errors=True)
shutil.rmtree(EMB_SHARD_DIR, ignore_errors=True)
print("All shard directories cleared.")

## 10. Save outputs (drop-in replacement for `backend/data/`)

In [ ]:
import json

faiss.write_index(index, os.path.join(OUT_DIR, "index.faiss"))

with open(os.path.join(OUT_DIR, "chunks.pkl"), "wb") as f:
    pickle.dump(all_chunks, f)

# One metadata entry per source file (gr_id + language), deduped from the chunk-level records —
# generated from the exact same run, so it can never drift out of sync with the index again.
seen = {}
for c in all_chunks:
    key = (c["gr_id"], c["language"])
    if key not in seen:
        seen[key] = {
            "gr_id": c["gr_id"],
            "department": c["department"],
            "language": c["language"],
            "title": c["title"],
            "gr_number": c["gr_number"],
            "issued_on": c["issued_on"],
        }

with open(os.path.join(OUT_DIR, "metadata.json"), "w", encoding="utf-8") as f:
    json.dump(list(seen.values()), f, ensure_ascii=False, indent=2)

print("Saved index.faiss, chunks.pkl, metadata.json to", OUT_DIR)

## 11. Sanity check

In [ ]:
print("Total chunks:", len(all_chunks))
print("Total vectors in index:", index.ntotal)
assert len(all_chunks) == index.ntotal, "chunks.pkl and index.faiss are out of alignment!"

unique_files_indexed = {(c["gr_id"], c["language"]) for c in all_chunks}
print("Unique gr_id+language pairs indexed:", len(unique_files_indexed))
print("Unique gr_id+language pairs discovered on disk:", len({(g, l) for _, g, _, l in discovered_files}))

parsed_titles = sum(1 for c in all_chunks if c["title"])
parsed_dates = sum(1 for c in all_chunks if c["issued_on"])
parsed_gr_no = sum(1 for c in all_chunks if c["gr_number"])
print(f"Chunks with parsed title:      {parsed_titles}/{len(all_chunks)}")
print(f"Chunks with parsed issued_on:  {parsed_dates}/{len(all_chunks)}")
print(f"Chunks with parsed gr_number:  {parsed_gr_no}/{len(all_chunks)}")

# Smoke-test retrieval
query_model = model if n_gpus < 2 else SentenceTransformer(MODEL_NAME)
q_emb = query_model.encode(["query: Dearness Allowance arrears"], convert_to_numpy=True).astype("float32")
faiss.normalize_L2(q_emb)
D, I = index.search(q_emb, 5)
for score, idx in zip(D[0], I[0]):
    c = all_chunks[idx]
    print(f"{score:.3f}  {c['gr_id']}  {c['title']}")

## 12. Zip for download

Zips `index.faiss`, `chunks.pkl`, and `metadata.json` from `OUT_DIR` into a single
`/kaggle/working/backend_data.zip`, ready to download from the notebook's **Output** tab. Each
file is deleted right after being added to the zip (rather than zipping the whole folder at once)
so the run never holds two full copies of a multi-GB index simultaneously — relevant given
`/kaggle/working` is capped at 20GB. Uses no compression (`ZIP_STORED`) since the FAISS index is
dense float32 data that doesn't meaningfully compress anyway — this keeps the zip step itself
fast.

In [ ]:
import zipfile

zip_path = "/kaggle/working/backend_data.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_STORED) as zf:
    for fname in os.listdir(OUT_DIR):
        fpath = os.path.join(OUT_DIR, fname)
        zf.write(fpath, arcname=fname)
        os.remove(fpath)

os.rmdir(OUT_DIR)

print(f"Saved {zip_path} — download it from the Kaggle notebook's Output tab.")
print("Unzip it and replace index.faiss, chunks.pkl, metadata.json under backend/data/ in the project.")